# BANKING77 Part 3: Structured JSON Output, Validation and Mock Tools
**Run:** Runtime -> Change runtime type -> **T4 GPU**, then Runtime -> Run all.

Flow tested here: `Customer request -> Qwen JSON action -> schema validation -> mock tool -> final response`.
All tools use mock data only; no real banking data or credentials.

## 1 - Get the project code and install dependencies

In [ ]:
import os
from google.colab import userdata

BRANCH = "2025paml541-Karthik"
REPO = "github.com/2025paml620/AppliedGenAI-Banking77-Project-Group2.git"

try:
    # Retrieve your GitHub token securely from Colab Secrets
    # Step 1: Create a classic token with complete access in GIT HUB/Setting/Developer settings/Personal access tokens/Token (classic)
    # Step 2: Add the token in Colab Secrets (🔑) on the left side with name GITHUB_TOKEN
    token = userdata.get('GITHUB_TOKEN')
    authenticated_repo = f"https://{token}@{REPO}"
except Exception as e:
    print("Error retrieving GITHUB_TOKEN. Make sure you added it in the Secrets tab (🔑) on the left.")
    authenticated_repo = f"https://{REPO}"

if not os.path.isdir("project"):
    !git clone -q --branch {BRANCH} {authenticated_repo} project

%cd project
%pip -q install "transformers==4.57.6" "datasets==3.6.0" "accelerate>=0.33" pandas pytest

## 2 - Unit tests (no model needed): valid and invalid JSON, tool failures, guardrails

In [ ]:
!python -m pytest tests/test_structured.py -v

## 3 - Prompt template and schema validation

In [ ]:
import json, pandas as pd
from run_part3 import load_intents, run_demo, DEMO_QUERIES
from src.prompts.structured import build_structured_prompt
from src.structured.schema import validate_output

intents = load_intents("data/evaluation_sample.csv")
print(len(intents), "labels")
print(build_structured_prompt("Where is TXN1002?", intents)[-900:])

In [ ]:
good = '{"intent": "lost_or_stolen_card", "response": "Please use the official app.", "action_required": false, "action": null}'
samples = {
    "valid": good,
    "markdown fence": "```json\n" + good + "\n```",
    "not json": "Sure! Your card is blocked.",
    "bad label": good.replace("lost_or_stolen_card", "made_up"),
    "missing key": '{"intent": "lost_or_stolen_card"}',
    "action mismatch": good.replace('"action_required": false', '"action_required": true'),
}
for name, raw in samples.items():
    data, errors = validate_output(raw, intents)
    print(f"{name:16} -> {'OK' if data else errors}")

## 4 - Load Qwen2.5-1.5B-Instruct

In [ ]:
from src.model.llm_baseline import load_model
model, tokenizer = load_model()

## 5 - End-to-end demo: tool calls, missing/unknown arguments, blocked input

In [ ]:
pd.set_option("display.max_colwidth", 90, "display.width", 250)
df = run_demo(model, tokenizer, intents)
display(df[["query", "status", "intent", "tool", "final_response"]])
print(df["status"].value_counts().to_dict())

Raw model output and validation errors (for failure analysis):

In [ ]:
for _, r in df.iterrows():
    print("Q:", r["query"]); print("  raw:", r["raw_output"]); print("  errors:", r["errors"] or "-"); print()

## 6 - Try your own query

In [ ]:
from src.model.llm_baseline import generate_response
from src.structured.pipeline import handle_query
gen = lambda p: generate_response(model, tokenizer, p, {"do_sample": False, "max_new_tokens": 200})
result = handle_query("Block my card ending 5678", intents, gen)
print(json.dumps(result, indent=2, default=str))

## 7 - Notes
- Failures never reach the user as raw model text: invalid JSON, unsafe text and tool errors all return fixed/explanatory messages.
- Tool-action replies are built from the tool result, not from model text.
- Small models may break the JSON format or omit an action; the status column shows how often (`invalid_output`).
- Adaptation comparison (prompting vs RAG vs PEFT): see `reports/PART3_REPORT.md`.